<a href="https://colab.research.google.com/github/meghanavanamala/predictiveanalysis/blob/recommand-news-article/recommand_news_article.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

# ----------------------------
# Sample user interaction data
# ----------------------------
user_article_data = {
    'User': ['User1', 'User1', 'User2', 'User2', 'User3', 'User3', 'User4'],
    'Article': ['Mystery1', 'Politics1', 'Politics1', 'Sports1', 'Mystery1', 'Tech1', 'Tech1'],
    'Interaction': [1, 1, 1, 1, 1, 1, 1]  # 1 means read/liked
}

df = pd.DataFrame(user_article_data)

# ----------------------------
# Article Metadata
# ----------------------------
article_metadata = {
    'Article': ['Mystery1', 'Politics1', 'Sports1', 'Tech1', 'Mystery2', 'Politics2', 'Sports2', 'Tech2'],
    'Tags': ['mystery', 'politics', 'sports', 'technology', 'mystery', 'politics', 'sports', 'technology']
}
articles = pd.DataFrame(article_metadata)

# ----------------------------
# Merge interaction with tags
# ----------------------------
df = df.merge(articles, on='Article')

# ----------------------------
# User Profile Creation
# ----------------------------
user_profiles = df.groupby(['User', 'Tags']).size().unstack(fill_value=0)

# ----------------------------
# Similarity Between Users
# ----------------------------
similarity = pd.DataFrame(
    cosine_similarity(user_profiles),
    index=user_profiles.index,
    columns=user_profiles.index
)

# ----------------------------
# Recommendation Function
# ----------------------------
def recommend_articles(user_id, top_n=3):
    if user_id not in user_profiles.index:
        return "User not found"

    # Get similar users
    similar_users = similarity[user_id].sort_values(ascending=False)[1:]  # exclude self
    user_articles = df[df['User'] == user_id]['Article'].unique()

    recommended = []

    for sim_user in similar_users.index:
        sim_user_articles = df[df['User'] == sim_user]['Article']
        for article in sim_user_articles:
            if article not in user_articles and article not in recommended:
                recommended.append(article)
            if len(recommended) >= top_n:
                break
        if len(recommended) >= top_n:
            break

    if not recommended:
        return "No new article recommendations available."

    return recommended

# ----------------------------
# Try it for a user
# ----------------------------
print("Recommended Articles for User1:", recommend_articles("User1"))
print("Recommended Articles for User2:", recommend_articles("User2"))
print("Recommended Articles for User4:", recommend_articles("User4"))


Recommended Articles for User1: ['Sports1', 'Tech1']
Recommended Articles for User2: ['Mystery1', 'Tech1']
Recommended Articles for User4: ['Mystery1', 'Politics1', 'Sports1']
